In [2]:
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

from torch.nn.utils.rnn import pad_sequence

### tokenize_prompt_and_output

In [ ]:
''' For this function, I modified the test snapshot to align with my solution, which slices input_ids before padding.'''
import numpy as np
data = dict(np.load('tests/_snapshots/test_tokenize_prompt_and_output.npz'))
data['input_ids'] = data['input_ids'].copy()
data['input_ids'][0][7] = 151643 # it used to be 0
np.savez('tests/_snapshots/test_tokenize_prompt_and_output.npz', **data)

''' to view content of a .npz file'''
data = np.load('tests/_snapshots/test_tokenize_prompt_and_output.npz')
for key in data.files:
    print(f"\n{key}:")
    print(data[key])

In [86]:
prompts = ["that was a dream!", "green earth!", "i can do it!"]
completions = ["it was a huge one", "blue sky", "hell, yeah"]
model_id = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_id)

prompt_tokens = tokenizer(prompts)['input_ids']
completion_tokens = tokenizer(completions)['input_ids']
prompt_completions = [p+c for p,c in zip(prompt_tokens, completion_tokens)]
prompt_lens = [len(p) for p in prompt_tokens]
total_lens = [len(pc) for pc in prompt_completions]
max_len = max(total_lens)
seq_len = max_len - 1
batch_size = len(prompt_completions)
pad_id = tokenizer.eos_token_id

input_ids = torch.full((batch_size, seq_len), pad_id, dtype=torch.long)
labels = torch.full((batch_size, seq_len), pad_id, dtype=torch.long)
response_mask = torch.full((batch_size, seq_len), False, dtype=torch.bool)

for idx, pc in enumerate(prompt_completions):
    t = torch.tensor(pc, dtype=torch.long)
    input_ids[idx, :total_lens[idx]-1] = t[:-1]
    labels[idx, :total_lens[idx]-1] = t[1:]
    response_mask[idx, prompt_lens[idx]-1:total_lens[idx]-1] = True

In [87]:
print(prompt_tokens)
print('----------------')
print(completion_tokens)
print('----------------')
print(input_ids)
print('----------------')
print(labels)
print('----------------')
print(response_mask)

[[5562, 373, 257, 4320, 0], [14809, 4534, 0], [72, 460, 466, 340, 0]]
----------------
[[270, 373, 257, 3236, 530], [17585, 6766], [12758, 11, 10194]]
----------------
tensor([[ 5562,   373,   257,  4320,     0,   270,   373,   257,  3236],
        [14809,  4534,     0, 17585, 50256, 50256, 50256, 50256, 50256],
        [   72,   460,   466,   340,     0, 12758,    11, 50256, 50256]])
----------------
tensor([[  373,   257,  4320,     0,   270,   373,   257,  3236,   530],
        [ 4534,     0, 17585,  6766, 50256, 50256, 50256, 50256, 50256],
        [  460,   466,   340,     0, 12758,    11, 10194, 50256, 50256]])
----------------
tensor([[False, False, False, False,  True,  True,  True,  True,  True],
        [False, False,  True,  True, False, False, False, False, False],
        [False, False, False, False,  True,  True,  True, False, False]])


In [ ]:
if tokenizer.pad_token is None:
    # Option A: Use the EOS token as the pad token (Common for LLMs)
    tokenizer.pad_token = tokenizer.eos_token
    # Option B: Add a brand new [PAD] token
    # tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    # Note: If you add a NEW token, you must also resize the model's embeddings:
    # model.resize_token_embeddings(len(tokenizer))
print(tokenizer.pad_token_id)

50256


In [61]:
'''this will not throw an error'''
t = torch.zeros((4))
t[:10] = 1 #!
print(t)

tensor([1., 1., 1., 1.])


In [ ]:
### tell about uv not sync and tokenizer size on disc

### get_response_log_probs

In [ ]:
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
B, T, V = 2, 5 ,len(tokenizer)
labels = torch.randint(0, V, (B, T))
labels = labels.unsqueeze(dim = -1)
logits = torch.randn((B, T, V))
g = logits.gather(dim = -1, index = labels)
print(g.shape)
g = g.squeeze(dim=-1)
print(g.shape)

torch.Size([2, 5, 1])
torch.Size([2, 5])


In [ ]:
input_ids = torch.randint(0, V, (B, T))
labels = torch.randint(0, V, (B, T))
model = AutoModelForCausalLM.from_pretrained(model_name)
labels = labels.unsqueeze(dim = -1)
out = model(input_ids)
log_probs = torch.log_softmax(out.logits, dim=-1)
response_log_probs = (log_probs.gather(dim = -1, index = labels)).squeeze(-1)

In [122]:
x = torch.randn((4,3))
dim = None
x.sum(dim=dim)

tensor(-4.8358)

In [ ]:
# In case you get torch.OutOfMemoryError: CUDA out of memory
# Chunk over sequence length to avoid materializing the full [batch, seq_len, vocab_size] log_probs tensor all at once.

def get_response_log_probs(
    model: PreTrainedModel,
    input_ids: torch.Tensor,
    labels: torch.Tensor,
    return_token_entropy: bool = False,
    chunk_size: int = 512,
    ) -> dict[str, torch.Tensor]:

    labels = labels.unsqueeze(dim=-1)
    logits = model(input_ids).logits

    # Chunk over sequence length to avoid materializing the full
    # [batch, seq_len, vocab_size] log_probs tensor all at once.
    seq_len = logits.shape[1]
    log_prob_chunks = []
    entropy_chunks = [] if return_token_entropy else None

    for start in range(0, seq_len, chunk_size):
        chunk = logits[:, start:start + chunk_size, :]
        log_p = torch.log_softmax(chunk, dim=-1)
        log_prob_chunks.append(
            log_p.gather(dim=-1, index=labels[:, start:start + chunk_size, :]).squeeze(-1)
        )
        if return_token_entropy:
            entropy_chunks.append(-(log_p.exp() * log_p).sum(dim=-1))

    del logits
    response_log_probs = torch.cat(log_prob_chunks, dim=1)

    if not return_token_entropy:
        return {"log_probs": response_log_probs}
    else:
        return {"log_probs": response_log_probs, "token_entropy": torch.cat(entropy_chunks, dim=1)}

### sft_microbatch_train_step

In [ ]:
from cs336_alignment.hw5 import masked_normalize

# policy_log_probs_grad = torch.tensor([
#     [-0.25, -0.25,  0.0, -0.25,  0.0, -0.25,  0.0, -0.25, -0.25,  0.0],
#     [-0.25, -0.25, -0.25, -0.25, -0.25,  0.0, -0.25, -0.25,  0.0, -0.25]
# ])
policy_log_probs = torch.randn(2,10)
response_mask = torch.rand(2, 10) > 0.5
normalize_constant = response_mask.sum()
gradient_accumulation_steps = 4
loss = masked_normalize(policy_log_probs, response_mask, normalize_constant)/gradient_accumulation_steps

# Incorrect: loss = - masked_normalize(policy_log_probs, response_mask, normalize_constant) / (response_mask.sum())
loss = - (masked_normalize(policy_log_probs, response_mask, normalize_constant, dim=1).mean()) / gradient_accumulation_steps

### compute_group_normalized_rewards

In [128]:
import hashlib
def dummy_reward_fn(response, ground_truth):
        # Use SHA-256 which is deterministic
        response_hash = int(hashlib.sha256(response.encode()).hexdigest(), 16)
        reward = (response_hash % 10) / 10.0
        return {
            "reward": reward,
            "format_reward": reward,
            "answer_reward": reward,
        }

rollout_responses = ['hmm I think ths answer is 0', 'hmm I think ths answer is 1', 'hmm I think ths answer is 2', 'hmm I think ths answer is 3', 'hmm I think ths answer is 4', 'hmm I think ths answer is 5', 'hmm I think ths answer is 6', 'hmm I think ths answer is 7']
repeated_ground_truths = ['42', '42', '42', '42', '42', '42', '42', '42']
group_size = 4

In [146]:
dummy_reward = dummy_reward_fn('what is 2+2?', '4 really')
print(dummy_reward)
results = [dummy_reward_fn(r, g) for r, g in zip(rollout_responses, repeated_ground_truths)]
rewards = torch.tensor([x['reward'] for x in results])
print(rewards)

{'reward': 0.4, 'format_reward': 0.4, 'answer_reward': 0.4}
tensor([0.2000, 0.5000, 0.0000, 0.4000, 0.9000, 0.7000, 0.2000, 0.0000])


In [ ]:
rewards_grouped = torch.tensor(rewards).view(-1, group_size)
mean = rewards_grouped.mean(dim=-1, keepdim=True)
std = rewards_grouped.std(dim=-1, keepdim=True)

### compute_grpo_clip_loss

In [147]:
t = torch.arange(12).view(3,4)
clipped_t = torch.clamp(t, min=3, max=9)
print(clipped_t)

tensor([[3, 3, 3, 3],
        [4, 5, 6, 7],
        [8, 9, 9, 9]])


In [151]:
a = torch.randint(0,9, (3,3))
b = torch.randint(0,9, (3,3))
c = torch.min(a, b)
print(a)
print(b)
print('----------')
print(c)


tensor([[6, 7, 2],
        [0, 3, 3],
        [1, 4, 7]])
tensor([[1, 3, 8],
        [5, 2, 5],
        [4, 5, 5]])
----------
tensor([[1, 3, 2],
        [0, 2, 3],
        [1, 4, 5]])


In [153]:
a.sum(dim=-1)
print(a)

tensor([[6, 7, 2],
        [0, 3, 3],
        [1, 4, 7]])


### Math Baseline

In [7]:
model_name = 'Qwen/Qwen2.5-Math-1.5B'
model_name = 'gpt2'
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:

prompts = ['what is 4+4?', 'who are you?']
inputs = tokenizer(prompts, return_tensors="pt", padding=True)
outputs = model.generate(**inputs, max_new_tokens=50)
completions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
completions = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(completions)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


In [ ]:
prompts = ['what is 4+4?', 'who are you?']
inputs = tokenizer(prompts, return_tensors="pt", padding=True)
outputs = model.generate(**inputs, max_new_tokens=50)
completions = tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [37]:
from cs336_alignment.drgrpo_grader import r1_zero_reward_fn

''' this pass'''
response = '<think> for so many reason </think> <answer> she has 0.25 </answer>'
ground_truth = '0.25'

''' this pass'''
response = '<think> for so many reason </think> <answer> 1/4 </answer>'
ground_truth = '0.25'

''' this fails'''
response = '<think> for so many reason </think> <answer> she has 1/4 </answer>'
ground_truth = '0.25'

ans = r1_zero_reward_fn(response, ground_truth, fast=True)
print(ans)

{'format_reward': 1.0, 'answer_reward': 0.0, 'reward': 0.0}


### Data Loader

In [ ]:
import json
data_file = 'data/sft/sft_gpt-oss-120b_filtered.jsonl'
with open(data_file) as f:
    data_list = json.load(f)
    # data = [json.loads(line) for line in f if line.strip()] for actual .jsonl file
print(type(data_list))
print(data_list[0])

<class 'list'>
{'problem': 'The two lines $y = 4x - 19$ and $2x+y = 95$ intersect. What is the value of $x$ at the point of intersection?', 'reasoning_trace': 'We substitute the expression for\u202fy from the first equation into the second: 2x\u202f+\u202f(4x\u202f−\u202f19)\u202f=\u202f95 → 6x\u202f−\u202f19\u202f=\u202f95 → 6x\u202f=\u202f114 → x\u202f=\u202f19.</think> <answer>19</answer>', 'extracted_answer': '19', 'expected_answer': '19'}


In [23]:
from pathlib import Path
import json
import numpy as np

class SFTDataLoader:
    def __init__(self, data_file, prmpt_file, batch_size, seed=42):
        self.data_file = data_file
        self.prmpt_file = prmpt_file
        self.batch_size = batch_size
        self.rng = np.random.RandomState(seed)

        self._load_data()
        self.reset()
    # def _load_data(self):
    #     with open(self.data_file) as f:
    #         self.data_list = json.load(f)
    def _load_data(self):
        with open(self.data_file) as f: data = json.load(f)
        template = Path("cs336_alignment/prompts/r1_zero.prompt").read_text()
        self.data_list = [
            {
                'prompt':template.format(question = e['problem']),
                'completion':e['reasoning_trace'],
                'ground_truth':e['expected_answer']
             }
             for e in data]

    def get_batch(self):
        batch_data = self.data_list[self.ptr: self.ptr + self.batch_size]
        self.ptr += self.batch_size
        if self.ptr >= len(self.data_list): self.reset()
        return batch_data
    def reset(self):
        self.rng.shuffle(self.data_list)
        self.ptr = 0

    def __iter__(self):
        self.reset() # every new for loop starts from the beginning with a fresh shuffle.
        return self

    def __next__(self):
        if self.ptr >= len(self.data_list):
            raise StopIteration
        batch = self.data_list[self.ptr: self.ptr + self.batch_size]
        self.ptr += self.batch_size
        return batch


In [24]:
data_file, prmpt_file, batch_size = 'data/sft/sft_gpt-oss-120b_filtered.jsonl', 'dummy', 4
train_data_loader = SFTDataLoader(data_file, prmpt_file, batch_size)

In [25]:
''' call get_batch() inside a for loop'''
for i in range(3):
    batch = train_data_loader.get_batch()
    print(batch)
    print('------------')

[{'prompt': 'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Find the area of the triangle with vertices $(-1,4),$ $(7,0),$ and $(11,5).$\nAssistant: <think>', 'completion': '\nTo find the area of a triangle given its vertices, we can use the determinant (shoelace) formula:  \n\n\\[\n\\text{Area} = \\frac12\\big|x_1(y_2-y_3)+x_2(y_3-y_1)+x_3(y_1-y_2)\\big|.\n\\]\n\nPlugging in the points \\((-1,4),\\;(7,0),\\;(11,5)\\):\n\n\\[\n\\begin{aligned}\nx_1(y_2-y_3) &= -1(0-5)=5,\\\\\nx_2(y_3-y_1) &= 7(5-4)=7,\\\\\nx_3(y_1-y_2) &= 11(4-0)=44.\n\\end{aligned}\n\\]\n\nSumming gives \\(5+7+44=56\\).  \nThus the area is \\(\\frac12

In [ ]:
''' call it as iterable'''
for i, e in enumerate(train_data_loader):
    print(e)
    if i == 3: break

[{'problem': 'Three coplanar squares with sides of lengths two, four and six units, respectively, are arranged side-by-side, as shown so that one side of each square lies on line $AB$ and a segment connects the bottom left corner of the smallest square to the upper right corner of the largest square. What is the area of the shaded quadrilateral? [asy] size(150); defaultpen(linewidth(0.9)+fontsize(10));\nfill((2,0)--(6,0)--(6,3)--(2,1)--cycle,gray(0.8));\ndraw(scale(2)*unitsquare);\ndraw(shift(2,0)*scale(4)*unitsquare);\ndraw(shift(6,0)*scale(6)*unitsquare);\ndraw((0,0)--(12,6));\n\nreal d = 1.2; pair d2 = (0.9,0);\npair A = (-d,0), B = (12+d,0); dot(A,linewidth(3)); dot(B,linewidth(3)); label("A",A,(0,-1.5)); label("B",B,(0,-1.5)); draw(A-d2--B+d2,Arrows(4));\nlabel("2",(1,2.7)); label("4",(4,4.7)); label("6",(9,6.7));\n[/asy]', 'reasoning_trace': '\nPlace the squares consecutively on the x‑axis: the smallest square from \\(x=0\\) to \\(x=2\\), the middle one from \\(x=2\\) to \\(x=6\\

In [20]:
class Iterable:
    def __init__(self):
        self.data_list = list(range(12))
        self.ptr = 0
        self.batch_size = 4

    def __iter__(self):
        self.ptr = 0 # every new for loop starts from the beginning
        return self

    def __next__(self):
        if self.ptr >= len(self.data_list):
            raise StopIteration
        batch = self.data_list[self.ptr: self.ptr + self.batch_size]
        self.ptr += self.batch_size
        return batch

iterable = Iterable()
for e in iterable:
    print(e)
    print('----------------')

[0, 1, 2, 3]
----------------
[4, 5, 6, 7]
----------------
[8, 9, 10, 11]
----------------


### SFT Training Loop

In [ ]:
    # ── Training loop ────────────────────────────────────────────────────────
    # steps = len(train_data_loader)//micro_batch_size
    # gradient_accumulation_steps =  batch_size //micro_batch_size
    # for step in range(1, steps+1):
    #     batch = train_data_loader.get_batch()
    #     prompts = [e['prompt'] for e in batch]
    #     completions  = [e['completion'] for e in batch]
    #     tokenized_batch = tokenize_prompt_and_output(prompts, completions, tokenizer)
    #     max_len = model.config.n_positions  # 1024 for GPT-2
    #     for key in tokenized_batch: tokenized_batch[key] = tokenized_batch[key][:, :max_len]
    #     input_ids, labels,  completion_mask = tokenized_batch["input_ids"], tokenized_batch["labels"], tokenized_batch["response_mask"]
    #     logps = get_response_log_probs(model, input_ids, labels)['log_probs']
    #     normalize_constant = completion_mask.sum(dim=1)
    #     loss, metadata = sft_microbatch_train_step(logps, completion_mask, gradient_accumulation_steps, normalize_constant)
    #     if not step%gradient_accumulation_steps:
    #         optimizer.step()
    #         # clip somewhere
    #         optimizer.zero_grad()

In [ ]:
# sampling_params = SamplingParams()
# reward_fn = r1_zero_reward_fn
# n_grpo_steps = 10



# # eval_metrics = evaluate_vllm()


# for grpo_step in range(n_grpo_steps):
#     cur_batch = random.sample(train_dataset, n_prompts_per_rollout_batch)
#     rep_rollout_prompts = [prompt_template.replace("{question}", ex['problem']) for ex in cur_batch
#                             for _ in range(group_size)]
#     rep_rollout_ground_truths = [ex['answer'] for ex in cur_batch for _ in range(group_size)]


#     rollout_outputs = vllm_model.generate(rep_rollout_prompts, sampling_params)
#     rollout_responses = [output.outputs[0].text for output in rollout_outputs]


#     rollout_advantages, rollout_raw_rewards, rollout_rewards_meta = compute_group_normalized_rewards(
#         reward_fn,
#         rollout_responses,
#         rep_rollout_ground_truths,
#         group_size,
#         advantage_eps,
#         use_std_normalization,
#     )


#     input_ids, labels, completion_mask = tokenize_prompt_and_output(rep_rollout_prompts, rollout_responses, tokenizer)
#     mean_response_length = completion_mask.sum(dim=1).float().mean().item()



#     model.eval()
#     old_log_probs = []
#     old_log_probs_train_size = 50 ## fix
#     device = 'cpu' ## fix
#     # compute old log probs for each microbatch
#     total_train_size, batch_size = len(input_ids), old_log_probs_train_size
#     for idx in range(0, len(input_ids), old_log_probs_train_size):
#         input_ids_idx, labels_idx = map(lambda x: x[idx:idx+batch_size].to(device), [input_ids, labels])
#         with torch.no_grad():
#             with torch.autocast(device_type=device.split(":")[0]):
#                 log_probs = get_response_log_probs(model, input_ids, labels)["log_probs"]
#             old_log_probs.append(log_probs)
#     # concatenate the old log probs for each microbatch and offload to CPU
#     old_log_probs = torch.cat(old_log_probs, dim=0).cpu()
    

#     # clear torch cache (to save memory)
#     torch.cuda.empty_cache()
#     model.train()

#     for train_epoch in range(epochs_per_rollout_batch):
#         num_train_steps = rollout_batch_size // train_batch_size
#         for train_step in range(num_train_steps):

#             loss_accum = 0.0
#             entropy_accum = 0.0
#             clip_fraction_accum = 0.0
#             mean_ratio_accum = 0.0
#             total_response_tokens = 0

#             # loop through microbatches
#             for idx in range(grad_acc_steps):

#                 # get the base index, start index, and end index for the microbatch
#                 base_idx = train_step * train_batch_size
#                 start_idx = base_idx + idx * micro_train_batch_size
#                 end_idx = start_idx + micro_train_batch_size

#                 # microbatch of input_ids, labels, and response_mask
#                 microbatch = {k: v[start_idx:end_idx].to(device) for k, v in tokenized_train_data.items()}

#                 # compute current policy log_probs and token_entropy for the microbatch
#                 with torch.autocast(device_type=device.split(":")[0], dtype=DTYPE_MAPPING[training.dtype]):
#                     cur_log_probs_result = get_response_log_probs(model, microbatch["input_ids"], microbatch["labels"], True)

#                 # calculate the loss for the microbatch
#                 current_log_probs = cur_log_probs_result["log_probs"]
#                 token_entropy = cur_log_probs_result["token_entropy"]
#                 old_log_probs_microbatch = old_log_probs[start_idx:end_idx].to(device) if old_log_probs is not None else None
#                 loss, meta = grpo_microbatch_train_step(
#                     policy_log_probs=current_log_probs,
#                     response_mask=microbatch["response_mask"],
#                     gradient_accumulation_steps=grad_acc_steps,
#                     loss_type=config.training.loss_type,
#                     raw_rewards=rollout_raw_rewards[start_idx:end_idx].unsqueeze(-1).to(device),
#                     advantages=rollout_advantages[start_idx:end_idx].unsqueeze(-1).to(device),
#                     old_log_probs=old_log_probs_microbatch,
#                     cliprange=config.training.cliprange,
#                     norm_mode=config.training.normalize_mode,
#                     norm_constant=config.training.normalize_constant,
#                 )

#                 # accumulate
#                 loss_accum += loss.detach()
#                 entropy_accum += (token_entropy * microbatch["response_mask"]).sum().detach()
#                 total_response_tokens += microbatch["response_mask"].sum().detach()
#                 clip_fraction_accum += meta.get("clip_fraction", 0.0)
#                 mean_ratio_accum += meta.get("mean_ratio", 0.0)
#                 del microbatch, cur_log_probs_result, current_log_probs, token_entropy, loss, old_log_probs_microbatch


#             avg_entropy = entropy_accum / total_response_tokens
#             avg_clip_fraction = clip_fraction_accum / config.training.gradient_accumulation_steps
#             avg_mean_ratio = mean_ratio_accum / config.training.gradient_accumulation_steps
#             grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config.training.max_grad_norm)

#             # update the model parameters
#             optimizer.step()
#             optimizer.zero_grad(set_to_none=True)

        

#             # calculate mean rewards for the rollout batch
#             step_mean_format_reward = 0.0
#             step_mean_answer_reward = 0.0
#             step_mean_reward = 0.0
#             rewards_len = len(rollout_rewards_meta['rewards'])
#             for r in rollout_rewards_meta['rewards']:
#                 step_mean_format_reward += r["format_reward"] / rewards_len
#                 step_mean_answer_reward += r["answer_reward"] / rewards_len
#                 step_mean_reward += r["reward"] / rewards_len

        

#             torch.cuda.empty_cache()



#     # wake vLLM before loading updated policy weights and next generation step
#     if config.training.use_vllm_sleep_mode:
#         vllm_model.wake_up()
#     # load the model weights
#     load_policy_into_vllm_instance(model, vllm_model)

#     # intermediate evaluation on a subset of val_dataset
#     if config.training.eval_interval > 0 and ((grpo_step+1) % config.training.eval_interval == 0 or is_last_step):
#         eval_step += 1
#         pretty_print(f"Running intermediate evaluation on {config.training.max_val_examples} val examples...", title=f"{grpo_step_title} - Intermediate Evaluation", is_sub_title=True)
#         eval_start = time.time()
#         eval_metrics = evaluate_vllm(
#             vllm_model=vllm_model,
#             reward_fn=reward_fn,
#             val_dataset=val_dataset,
#             prompt_template=prompt_template,
#             sampling_params=sampling_params,
#             max_val_examples=config.training.max_val_examples,
#         )

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = 'Qwen/Qwen2.5-Math-1.5B'
# padding_side="left"
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

prompts = ["what is 2+2?", "Hi", "tell me about everything you know and beyond"]

inputs = tokenizer(prompts, return_tensors="pt", padding=True)
outputs = model.generate(**inputs, max_new_tokens=20)

decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(decoded)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


["what is 2+2? I've heard that 2+2=4, and that 2+2=5,", "Hi, I'm having trouble understanding how to find the equation of a line that is perpendicular to another line", 'tell me about everything you know and beyond.']


In [87]:
# print(tokenizer.pad_token) -> <|endoftext|>
# tokenizer.encode('<|endoftext|>') -> 151643
# tokenizer.decode([151643]) -> <|endoftext|>

'''
inputs = {'input_ids': B*T, 'attention_mask': B*T} (T = max(len(prompts)))
outputs = B*T (T = max(len(prompts)) + max(len(completions)))
'''

"\ninputs = {'input_ids': B*T, 'attention_mask': B*T} (T = max(len(prompts)))\noutputs = B*T (T = max(len(prompts)) + max(len(completions)))\n"

In [ ]:
# How to get prompt_lens
prompt_input_ids  = inputs["input_ids"]          # (N, P)
prompt_attn_mask  = inputs["attention_mask"]     # (N, P)
prompt_lens    = prompt_attn_mask.sum(-1)        # (N,)  true token counts

In [110]:
output_lens = (outputs != tokenizer.pad_token_id).int().sum(dim=1)
prompt_lens = (inputs['input_ids'] != tokenizer.pad_token_id).int().sum(dim=1)
completion_lens = output_lens - prompt_lens
print(completion_lens)

tensor([20, 20,  1])


In [ ]:
def tokenize_prompt_and_output_(prompts, tokenizer, model, max_new_tokens): # other relevant params here

    assert tokenizer.padding_side == "left", "Must use left-padding for batch generation"

    inputs = tokenizer(prompts, return_tensors="pt", padding=True)
    input_len = inputs['input_ids'].shape[1] # prompt + padding #
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

    prompt_lens = (inputs['input_ids'] != tokenizer.pad_token_id).int().sum(dim=1)
    total_lens = (outputs != tokenizer.pad_token_id).int().sum(dim=1)
    completion_lens = total_lens - prompt_lens

    seq_len = max(total_lens).item() - 1 
    batch_size = len(prompts)
    pad_id = tokenizer.pad_token_id

    input_ids = torch.full((batch_size, seq_len), pad_id, dtype=torch.long)
    labels = torch.full((batch_size, seq_len), pad_id, dtype=torch.long)
    response_mask = torch.full((batch_size, seq_len), False, dtype=torch.bool)

    for idx in range(batch_size):
        input_ids[idx, :total_lens[idx]-1] = outputs[idx, input_len - prompt_lens[idx]: input_len + completion_lens[idx] - 1]
        labels[idx, :total_lens[idx]-1] = outputs[idx, input_len - prompt_lens[idx] + 1: input_len + completion_lens[idx]]
        response_mask[idx, prompt_lens[idx]-1:total_lens[idx]-1] = True

    return {
        "input_ids": input_ids,
        "labels": labels,
        "response_mask": response_mask,
    } 

In [ ]:
tokenized_batch = tokenize_prompt_and_output_(prompts, tokenizer, model, 20)
input_ids, labels,  completion_mask = tokenized_batch["input_ids"], tokenized_batch["labels"], tokenized_batch["response_mask"]

### GRPO Training Loop

In [4]:
import random

from cs336_alignment.utils import load_dataset

In [3]:
train_data_file = 'data/grpo/train.jsonl'
train_dataset = load_dataset(data_file=train_data_file, data_type='train')

In [ ]:
n_prompts_per_rollout_batch = 8
group_size = 2
batch = random.sample(train_dataset, n_prompts_per_rollout_batch)
ground_truths = [e['answer'] for e in batch for _ in range(group_size)]

### OmegaConf

In [ ]:
from omegaconf import OmegaConf
from cs336_alignment.configs.defaults import Config # Config is a class that call others

In [30]:
config = OmegaConf.structured(Config) # dics of dics
print(config)
print(config.keys()) ## only 3 that are difine under class Config
prmpt_file = config.paths.prompt_template_file
print(prmpt_file)

{'paths': {'train_data_file': '/home/aayush/DATA/GRPO/train.jsonl', 'val_data_file': '/home/aayush/DATA/GRPO/validation.jsonl', 'prompt_template_file': 'cs336_alignment/prompts/r1_zero.prompt', 'model_path': '/home/aayush/DATA/Qwen2.5-Math-1.5B', 'output_dir': '/home/aayush/RESULTS/GRPO/run'}, 'vllm': {'gpu_memory_utilization': 0.2, 'dtype': 'bfloat16', 'enable_prefix_caching': True}, 'training': {'seed': 1337, 'device': 'cuda:0', 'dtype': 'bfloat16', 'attention_type': 'flash_attention_2', 'use_compile': False, 'learning_rate': 1e-05, 'weight_decay': 0.0, 'adam_beta1': 0.9, 'adam_beta2': 0.95, 'adam_eps': 1e-08, 'temperature': 1.0, 'top_p': 1.0, 'min_tokens': 4, 'max_tokens': 1024, 'stop': ['</answer>'], 'include_stop_str_in_output': True, 'n_grpo_steps': 10, 'advantage_eps': 1e-06, 'rollout_batch_size': 256, 'group_size': 8, 'epochs_per_rollout_batch': 1, 'train_batch_size': 256, 'grad_acc_steps': 256, 'loss_type': 'grpo_no_clip', 'use_std_normalization': True, 'cliprange': 0.2, 'max_

### Weights & Biases

In [7]:
import wandb

In [11]:
import random

import wandb

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="aakandeh",
    name= "blue-sky",
    tags = ["sun", "moon", "stars"],
    # Set the wandb project where this run will be logged.
    project="cs336-hw5",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)

# Simulate training.
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    # acc = 1 - 2**-epoch - random.random() / epoch - offset
    # loss = 2**-epoch + random.random() / epoch + offset
    mtr = 3**-epoch + random.random() / epoch + offset

    # Log metrics to wandb.
    # run.log({"acc": acc, "loss": loss})
    run.log({"mtr": mtr,})

# Finish the run and upload any remaining data.
run.finish()

mtr,█▄▃▂▇▄▃▁
mtr,0.06948


use `define_metric` to control how curves are plotted.

In [ ]:
wandb.init(
    project=config.training.wandb_project,
    name=config.training.wandb_run_name or None,
    config=config_dict,
    tags=config.training.wandb_tags,
        )
wandb.define_metric("grpo_step")
wandb.define_metric("eval_step")
wandb.define_metric("train/*", step_metric="grpo_step")
wandb.define_metric("eval/*", step_metric="eval_step")


'''initial evaluation snapshot'''
wandb.log({
    "eval/reward": eval_metrics['mean_reward'],
    "eval/answer_reward": eval_metrics['mean_answer_reward'],
    "eval/format_reward": eval_metrics['mean_format_reward'],
    "eval_step": eval_step,
    "grpo_step": 0,
})

'''training step logging'''
wandb.log({
    "train/loss": loss_accum.item(),
    "train/entropy": avg_entropy.item(),
    "train/reward": step_mean_reward,
    "train/answer_reward": step_mean_answer_reward,
    "train/format_reward": step_mean_format_reward,
    "train/clip_fraction": avg_clip_fraction,
    "train/mean_ratio": avg_mean_ratio,
    "train/grad_norm": grad_norm.item(),
    "train/mean_response_length": mean_response_length,
    "grpo_step": grpo_step,
})

'''evaluation after training update'''
wandb.log({
    "eval/reward": eval_metrics['mean_reward'],
    "eval/answer_reward": eval_metrics['mean_answer_reward'],
    "eval/format_reward": eval_metrics['mean_format_reward'],
    "eval_step": eval_step,
    "grpo_step": grpo_step,
})

'''timing instrumentation'''
wandb.log({
    "train/rollout_dt": rollout_dt, 
    "train/train_dt": train_dt, 
    "train/eval_dt": eval_dt, 
    "train/step_dt": step_dt, 
    "grpo_step": grpo_step
})

wandb.finish()